In [ ]:
import hashlib
import importlib
import json
import platform
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

NOTEBOOK = "06_walk_forward_evaluation.ipynb"
notebook_started = time.perf_counter()

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (
        path for path in candidates
        if (path / "pyproject.toml").is_file()
        and (path / "configs/research_config.example.json").is_file()
        and (path / "src/eurusd_research/regimes.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    entered = input("Full repository path containing configs and src: ").strip().strip('"')
    if not entered:
        raise ValueError("Enter the full repository path.")
    REPO_ROOT = Path(entered).expanduser().resolve()

if not (REPO_ROOT / "src/eurusd_research/regimes.py").is_file():
    raise RuntimeError("The selected repository lacks Notebook 05's regime module.")

import numpy as np
import pandas as pd

from eurusd_research.config import load_config
from eurusd_research import features as feature_io
from eurusd_research.data import _sha256
from eurusd_research.validation import utc, validate_plan

config = load_config(REPO_ROOT)
feature_settings = config["sampling_features"]
model_settings = config["baseline_models"]
regime_settings = config["liquidity_regimes"]
plan = validate_plan(config["pilot_validation_plan"])
HORIZONS = config["labels_validation"]["horizons_seconds"]

if HORIZONS != [1, 5, 30, 60]:
    raise RuntimeError("Expected horizons [1, 5, 30, 60].")

feature_manifest_path = (REPO_ROOT / config["active_features_manifest"]).resolve()
label_manifest_path = (REPO_ROOT / config["active_labels_manifest"]).resolve()
baseline_manifest_path = (REPO_ROOT / config["active_baseline_manifest"]).resolve()
regimes_manifest_path = (REPO_ROOT / config["active_regimes_manifest"]).resolve()

feature_state = json.loads(feature_manifest_path.read_text(encoding="utf-8"))
label_state = json.loads(label_manifest_path.read_text(encoding="utf-8"))
baseline_run = json.loads(baseline_manifest_path.read_text(encoding="utf-8"))
regimes_run = json.loads(regimes_manifest_path.read_text(encoding="utf-8"))

if any(state["status"] != "complete" for state in (
    feature_state, label_state, baseline_run, regimes_run
)):
    raise RuntimeError("Completed input generations are required.")
if feature_state["identity"]["settings"] != feature_settings:
    raise RuntimeError("Feature settings differ from the saved generation.")
if model_settings != baseline_run["identity"]["settings"]:
    raise RuntimeError("Keep the existing baseline model specification unchanged.")
if regime_settings != regimes_run["identity"]["settings"]:
    raise RuntimeError("Keep the existing regime specification unchanged.")
if label_state["identity"]["plan"] != plan:
    raise RuntimeError("The original pilot split plan has changed.")

input_hashes = {
    "features": _sha256(feature_manifest_path),
    "labels": _sha256(label_manifest_path),
}
if input_hashes != baseline_run["identity"]["input_manifest_sha256"]:
    raise RuntimeError("Baseline input manifests have changed.")
if label_state["identity"]["input_manifest_sha256"][0] != input_hashes["features"]:
    raise RuntimeError("Feature-to-label provenance differs.")

# Resolve runtime locations without rewriting historical manifests.
feature_state = {
    **feature_state,
    "output_dir": str(
        REPO_ROOT / "data/processed/features" / feature_state["job_id"]
    ),
}
label_state = {
    **label_state,
    "output_dir": str(
        REPO_ROOT / "data/processed/labels" / label_state["job_id"]
    ),
}

protected_start = utc(plan["periods"][2]["start_utc"])
available_start = utc(feature_state["identity"]["start_utc"])

wf_settings = config.get("walk_forward", {
    "training_calendar_days": 3,
    "evaluation_calendar_days": 1,
    "minimum_training_price_rows": 1000,
    "minimum_evaluation_price_rows": 1000,
    "max_folds": 31,
    "bootstrap_repetitions": 1000,
    "bootstrap_block_dates": [2, 3],
    "minimum_bootstrap_dates": 10,
    "confidence": 0.95,
    "random_seed": 42,
})

if wf_settings["training_calendar_days"] * 86400 > model_settings["max_rows_per_split"]:
    raise RuntimeError("The training window exceeds the existing row guard.")

versions = {
    name: version(name)
    for name in (
        "numpy", "pandas", "pyarrow", "duckdb",
        "scikit-learn", "scipy", "joblib", "pytest", "matplotlib",
    )
}
new_sources = {}

print("NOTEBOOK_06_PREFLIGHT")
print("Repository:", REPO_ROOT)
print("Feature generation:", feature_state["job_id"])
print("Label generation:", label_state["job_id"])
print("Protected holdout begins:", protected_start.isoformat())
print("Training calendar days:", wf_settings["training_calendar_days"])
print("Evaluation calendar days:", wf_settings["evaluation_calendar_days"])
print("No package installation or restart is required.")

In [ ]:
new_sources["src/eurusd_research/walk_forward.py"] = r'''
"""Chronological development folds, resumable fitting, and paired block intervals."""

import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from . import features as io
from .data import _sha256, _writer_lock
from .evaluation import training_bucket_cuts
from .models import (
    fit_ridge, load_development_window, predict_ridge, save_pipeline,
)
from .regimes import (
    build_liquidity_metrics, classify_regimes,
    fit_regime_thresholds, load_regime_inputs,
)
from .validation import utc


MODEL_NAMES = ("zero", "trailing", "ridge_price", "ridge_price_size")


def validate_fold(fold, protected_start):
    train_start = utc(fold["train_start_utc"])
    train_end = utc(fold["train_end_utc"])
    evaluation_start = utc(fold["evaluation_start_utc"])
    evaluation_end = utc(fold["evaluation_end_utc"])
    if not (
        train_start < train_end <= evaluation_start
        < evaluation_end <= utc(protected_start)
    ):
        raise ValueError("Fold chronology or protected holdout boundary is invalid.")
    return fold


def make_fold_plan(daily, available_start, protected_start, settings):
    available_start, protected_start = utc(available_start), utc(protected_start)
    days = settings["training_calendar_days"]
    if type(days) is not int or days < 1:
        raise ValueError("training_calendar_days must be a positive integer.")
    if settings["evaluation_calendar_days"] != 1:
        raise ValueError("This pilot uses nonoverlapping one-day evaluation windows.")

    q = daily[["date_utc", "usable_price_rows"]].copy()
    q["date_utc"] = pd.to_datetime(q["date_utc"], utc=True).dt.normalize()
    if not q["date_utc"].is_unique:
        raise ValueError("Daily coverage contains duplicate dates.")
    q = q.loc[
        q["date_utc"].ge(available_start)
        & q["date_utc"].lt(protected_start)
    ].sort_values("date_utc")
    active = q.loc[q["usable_price_rows"].gt(0), "date_utc"]
    if active.empty:
        raise ValueError("No usable development dates are available.")

    first_evaluation = active.iloc[0] + pd.Timedelta(days=days)
    folds, skipped = [], []
    for date in pd.date_range(
        first_evaluation, protected_start, freq="D", inclusive="left"
    ):
        train_start = date - pd.Timedelta(days=days)
        evaluation_end = date + pd.Timedelta(days=1)
        if evaluation_end > protected_start:
            continue
        training_rows = int(q.loc[
            q["date_utc"].ge(train_start) & q["date_utc"].lt(date),
            "usable_price_rows",
        ].sum())
        evaluation_rows = int(q.loc[
            q["date_utc"].eq(date), "usable_price_rows"
        ].sum())
        reason = None
        if training_rows < settings["minimum_training_price_rows"]:
            reason = "insufficient_training_price_coverage"
        elif evaluation_rows < settings["minimum_evaluation_price_rows"]:
            reason = "insufficient_evaluation_price_coverage"
        if reason:
            skipped.append({"date_utc": date.isoformat(), "reason": reason})
            continue

        fold = {
            "fold_id": date.strftime("wf_%Y%m%d"),
            "train_start_utc": train_start.isoformat(),
            "train_end_utc": date.isoformat(),
            "evaluation_start_utc": date.isoformat(),
            "evaluation_end_utc": evaluation_end.isoformat(),
        }
        folds.append(validate_fold(fold, protected_start))

    if not folds:
        raise ValueError("No eligible walk-forward folds were found.")
    if len(folds) > settings["max_folds"]:
        raise ValueError("Fold count exceeds the explicitly configured run limit.")
    return {"folds": folds, "skipped_evaluation_dates": skipped}


def fold_eligibility(frame, horizon, start, end, required_size_columns):
    start, end = utc(start), utc(end)
    timestamp = frame["feature_timestamp_utc"]
    target = frame[f"target_timestamp_utc_{horizon}s"]
    if not target.eq(timestamp + pd.Timedelta(seconds=horizon)).all():
        raise ValueError("Stored target timestamps disagree with their horizon.")

    inside = timestamp.ge(start) & timestamp.lt(end)
    valid = (
        inside & frame["price_valid"] & frame[f"label_valid_{horizon}s"]
        & np.isfinite(frame[f"label_{horizon}s_pips"])
    )
    before_end = target.lt(end)
    trailing = np.isfinite(frame[f"mid_change_{horizon}s_pips"])
    price = valid & before_end & trailing
    size_available = (
        frame["size_pair_valid"]
        & np.isfinite(
            frame[required_size_columns].to_numpy(dtype=float)
        ).all(axis=1)
    )
    common = price & size_available
    return common, {
        "clock_rows": int(inside.sum()),
        "price_valid_rows": int((inside & frame["price_valid"]).sum()),
        "valid_label_rows": int((inside & frame[f"label_valid_{horizon}s"]).sum()),
        "boundary_purged_rows": int((valid & ~before_end).sum()),
        "price_eligible_rows": int(price.sum()),
        "matched_rows": int(common.sum()),
        "price_rows_without_size": int((price & ~size_available).sum()),
        "price_rows_missing_trailing": int((valid & before_end & ~trailing).sum()),
    }


def _record(root, path, kind, metadata=None, **extra):
    path = Path(path).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError("An output path escapes the repository.")
    values = dict(metadata) if metadata is not None else {
        "bytes": path.stat().st_size, "sha256": _sha256(path)
    }
    return {
        **values, "kind": kind,
        "path": path.relative_to(root).as_posix(), **extra,
    }


def _verify_files(root, records):
    root = Path(root).resolve()
    for record in records:
        path = (root / record["path"]).resolve()
        if (
            not path.is_relative_to(root) or not path.is_file()
            or path.stat().st_size != record["bytes"]
            or _sha256(path) != record["sha256"]
        ):
            raise RuntimeError(f"Cached output integrity failed: {record['path']}")
        if "rows" in record and pq.ParquetFile(path).metadata.num_rows != record["rows"]:
            raise RuntimeError("Cached forecast row count changed.")
'''.lstrip()

print("Prepared fold planning, temporal eligibility, and integrity helpers.")

In [ ]:
new_sources["src/eurusd_research/walk_forward.py"] += r'''


def run_fold(repository_root, feature_state, label_state, fold, run_id, protocol):
    root = Path(repository_root).resolve()
    manifest_dir = root / "artifacts/manifests/walk_forward" / run_id
    with _writer_lock(manifest_dir):
        return _run_fold(
            root, feature_state, label_state, fold, run_id, protocol, manifest_dir
        )


def _run_fold(root, feature_state, label_state, fold, run_id, protocol, manifest_dir):
    validate_fold(fold, protocol["protected_start_utc"])
    began = time.perf_counter()
    manifest = manifest_dir / f"{fold['fold_id']}.json"
    identity = {
        "run_id": run_id,
        "protocol_sha256": hashlib.sha256(
            json.dumps(protocol, sort_keys=True).encode()
        ).hexdigest(),
        "fold": fold,
    }
    if manifest.exists():
        previous = json.loads(manifest.read_text(encoding="utf-8"))
        if previous["identity"] != identity:
            raise RuntimeError("Existing fold identity differs.")
        if previous["status"] == "complete":
            _verify_files(root, previous["files"])
            return {**previous, "cache_hit": True}

    state = {
        "status": "running", "identity": identity,
        "fold_id": fold["fold_id"], "files": [],
        "coverage": [], "fit_records": [], "bucket_cuts": {},
        "manifest_path": manifest.relative_to(root).as_posix(),
    }
    io.write_json(manifest, state)

    model_settings = protocol["model_settings"]
    regime_settings = protocol["regime_settings"]
    feature_settings = protocol["feature_settings"]
    horizons = protocol["horizons_seconds"]
    price_columns = model_settings["price_features"]
    all_columns = price_columns + model_settings["size_features"]
    load_columns = list(dict.fromkeys(
        all_columns + ["london_session", "new_york_session"]
    ))
    scope_bounds = {
        "train": (fold["train_start_utc"], fold["train_end_utc"]),
        "evaluation": (
            fold["evaluation_start_utc"], fold["evaluation_end_utc"]
        ),
    }

    frames = {}
    for role, (start, end) in scope_bounds.items():
        frames[role] = load_development_window(
            feature_state, label_state, start, end, load_columns, horizons,
            feature_settings, model_settings["max_rows_per_split"],
            model_settings["max_frame_mib"],
            manifest_dir / "loading" / fold["fold_id"] / role,
        )
    frame_bytes = {
        role: int(frame.memory_usage(index=True, deep=True).sum())
        for role, frame in frames.items()
    }
    if sum(frame_bytes.values()) > model_settings["max_combined_frame_mib"] * 1024**2:
        raise RuntimeError("Combined fold frames exceed the memory guard.")

    regime_metrics = {}
    for role, (start, end) in scope_bounds.items():
        raw, _ = load_regime_inputs(
            feature_state, root, start, end, regime_settings, feature_settings
        )
        regime_metrics[role] = build_liquidity_metrics(
            raw, start, end, regime_settings
        )
        del raw
    thresholds = fit_regime_thresholds(
        regime_metrics["train"], regime_settings, *scope_bounds["train"]
    )
    if utc(thresholds["fit_end_utc_exclusive"]) > utc(fold["evaluation_start_utc"]):
        raise RuntimeError("Regime fitting reaches into evaluation.")
    evaluation_regimes = classify_regimes(regime_metrics["evaluation"], thresholds)
    regime_lookup = evaluation_regimes.set_index(
        "feature_timestamp_utc"
    )["liquidity_regime"]
    del regime_metrics

    threshold_path = manifest_dir / f"{fold['fold_id']}-thresholds.json"
    io.write_json(threshold_path, thresholds)
    state["files"].append(_record(root, threshold_path, "thresholds"))
    state["thresholds"] = thresholds
    state["observed_input_frame_bytes"] = frame_bytes

    model_dir = root / "artifacts/models/walk_forward" / run_id / fold["fold_id"]
    forecast_dir = root / "artifacts/forecasts/walk_forward" / run_id / fold["fold_id"]

    for h in horizons:
        masks = {}
        for role, bounds in scope_bounds.items():
            mask, counts = fold_eligibility(
                frames[role], h, *bounds,
                model_settings["required_current_size_features"],
            )
            masks[role] = mask
            state["coverage"].append({
                "fold_id": fold["fold_id"], "horizon_seconds": h,
                "role": role, **counts,
            })
        train = frames["train"].loc[masks["train"]]
        evaluation = frames["evaluation"].loc[masks["evaluation"]]
        if min(len(train), len(evaluation)) < model_settings["minimum_fit_rows"]:
            raise RuntimeError(
                f"Too few matched rows in {fold['fold_id']} at {h}s. "
                "Stop rather than silently dropping this fold."
            )

        y_train = train[f"label_{h}s_pips"].to_numpy(dtype=float)
        train_predictions = {
            "zero": np.zeros(len(train)),
            "trailing": train[f"mid_change_{h}s_pips"].to_numpy(dtype=float),
        }
        predictions = {
            "zero": np.zeros(len(evaluation)),
            "trailing": evaluation[f"mid_change_{h}s_pips"].to_numpy(dtype=float),
        }
        for name, columns in (
            ("ridge_price", price_columns),
            ("ridge_price_size", all_columns),
        ):
            fitted = fit_ridge(
                train, y_train, columns,
                alpha=model_settings["ridge_alpha"],
                minimum_rows=model_settings["minimum_fit_rows"],
            )
            train_predictions[name] = predict_ridge(fitted, train)
            predictions[name] = predict_ridge(fitted, evaluation)
            path = model_dir / f"{name}-{h}s.joblib"
            saved = save_pipeline(fitted, path)
            state["files"].append(_record(
                root, path, "model", saved, horizon_seconds=h, model=name
            ))

        output = evaluation[[
            "feature_timestamp_utc", "source_timestamp_utc",
            "snapshot_staleness_seconds", "spread_pips",
            f"target_timestamp_utc_{h}s", f"label_{h}s_pips",
            f"same_source_record_{h}s",
            "london_session", "new_york_session",
        ]].rename(columns={
            f"target_timestamp_utc_{h}s": "target_timestamp_utc",
            f"label_{h}s_pips": "actual_change_pips",
            f"same_source_record_{h}s": "same_source_record",
        }).reset_index(drop=True)

        output["fold_id"] = pd.Series(
            fold["fold_id"], index=output.index, dtype="string"
        )
        output["horizon_seconds"] = h
        output["year_utc"] = output["feature_timestamp_utc"].dt.year.astype("int64")
        london, new_york = output["london_session"], output["new_york_session"]
        output["session_label"] = pd.Series(np.select(
            [london & new_york, london, new_york],
            ["london_new_york_overlap", "london_only", "new_york_only"],
            default="outside_defined_sessions",
        ), dtype="string")
        age = output["snapshot_staleness_seconds"]
        output["snapshot_age_band"] = pd.Series(np.select(
            [age.le(1), age.le(5)],
            ["0_to_1s", "over_1_to_5s"],
            default="over_5_to_30s",
        ), dtype="string")
        output["liquidity_regime"] = output["feature_timestamp_utc"].map(
            regime_lookup
        ).astype("string")

        if (
            output["liquidity_regime"].isna().any()
            or output["liquidity_regime"].eq("unavailable").any()
            or age.lt(0).any()
            or age.gt(feature_settings["max_staleness_seconds"]).any()
            or (output["source_timestamp_utc"] > output["feature_timestamp_utc"]).any()
        ):
            raise RuntimeError("Forecast state or regime alignment failed.")

        for name in MODEL_NAMES:
            if not np.isfinite(predictions[name]).all():
                raise RuntimeError("Nonfinite forecast encountered.")
            output[f"prediction_{name}_pips"] = predictions[name]
            cuts = training_bucket_cuts(
                train_predictions[name], model_settings["forecast_buckets"]
            )
            output[f"bucket_{name}"] = np.searchsorted(
                cuts, predictions[name], side="right"
            ).astype("int64")
            state["bucket_cuts"][f"{h}s/{name}"] = cuts.tolist()

        state["fit_records"].append({
            "horizon_seconds": h,
            "training_rows": len(train),
            "evaluation_rows": len(evaluation),
            "training_max_target_utc": train[f"target_timestamp_utc_{h}s"].max().isoformat(),
            "evaluation_max_target_utc": evaluation[f"target_timestamp_utc_{h}s"].max().isoformat(),
        })
        path = forecast_dir / f"{h}s.parquet"
        saved = io._save_frame(output, path)
        state["files"].append(_record(
            root, path, "forecasts", saved, horizon_seconds=h
        ))
        io.write_json(manifest, state)
        del fitted, train, evaluation, output

    state["runtime_seconds_when_fitted"] = round(time.perf_counter() - began, 3)
    state["status"] = "complete"
    io.write_json(manifest, state)
    return {**state, "cache_hit": False}
'''

print("Prepared resumable, training-only fold fitting.")

In [ ]:
new_sources["src/eurusd_research/walk_forward.py"] += r'''


def paired_block_interval(daily, block_dates=2, repetitions=1000,
                          confidence=0.95, seed=42, minimum_dates=10):
    columns = [
        "rows", "candidate_abs_sum", "reference_abs_sum",
        "candidate_sq_sum", "reference_sq_sum",
    ]
    q = daily.sort_values("date_utc").reset_index(drop=True)
    dates = pd.to_datetime(q["date_utc"], utc=True)
    values = q[columns].to_numpy(dtype=float)
    if (
        len(q) == 0 or dates.isna().any() or not dates.is_unique
        or not np.isfinite(values).all()
        or (values[:, 0] <= 0).any() or (values[:, 1:] < 0).any()
    ):
        raise ValueError("Valid paired daily loss totals are required.")
    if (
        type(block_dates) is not int or block_dates < 1
        or type(repetitions) is not int or repetitions < 20
        or not 0 < confidence < 1
    ):
        raise ValueError("Invalid bootstrap configuration.")

    total = values.sum(axis=0)
    point_mae = (total[1] - total[2]) / total[0]
    point_rmse = np.sqrt(total[3] / total[0]) - np.sqrt(total[4] / total[0])
    result = {
        "evaluation_dates": len(q),
        "block_dates": block_dates,
        "repetitions": repetitions,
        "confidence": confidence,
        "point_delta_mae_pips": float(point_mae),
        "point_delta_rmse_pips": float(point_rmse),
        "mae_interval_low_pips": None,
        "mae_interval_high_pips": None,
        "rmse_interval_low_pips": None,
        "rmse_interval_high_pips": None,
        "status": "insufficient_dates",
    }
    if len(q) < minimum_dates or len(q) < block_dates:
        return result

    rng = np.random.default_rng(seed)
    blocks_needed = math.ceil(len(q) / block_dates)
    starts = rng.integers(
        0, len(q) - block_dates + 1,
        size=(repetitions, blocks_needed),
    )
    indices = (
        starts[:, :, None] + np.arange(block_dates)
    ).reshape(repetitions, -1)[:, :len(q)]
    totals = values[indices].sum(axis=1)

    delta_mae = (totals[:, 1] - totals[:, 2]) / totals[:, 0]
    delta_rmse = (
        np.sqrt(totals[:, 3] / totals[:, 0])
        - np.sqrt(totals[:, 4] / totals[:, 0])
    )
    tail = (1 - confidence) / 2
    mae_bounds = np.quantile(delta_mae, [tail, 1 - tail])
    rmse_bounds = np.quantile(delta_rmse, [tail, 1 - tail])
    result.update({
        "mae_interval_low_pips": float(mae_bounds[0]),
        "mae_interval_high_pips": float(mae_bounds[1]),
        "rmse_interval_low_pips": float(rmse_bounds[0]),
        "rmse_interval_high_pips": float(rmse_bounds[1]),
        "status": "computed_exploratory",
    })
    return result
'''

print("Prepared paired moving-block intervals.")

In [ ]:
new_sources["tests/test_walk_forward.py"] = r'''
import numpy as np
import pandas as pd
import pytest

from eurusd_research.walk_forward import (
    _record, _verify_files, fold_eligibility,
    make_fold_plan, paired_block_interval, validate_fold,
)


def test_fold_dates_are_chronological_and_protected():
    dates = pd.date_range("2020-01-01", periods=15, tz="UTC")
    daily = pd.DataFrame({"date_utc": dates, "usable_price_rows": 200})
    settings = {
        "training_calendar_days": 3, "evaluation_calendar_days": 1,
        "minimum_training_price_rows": 10,
        "minimum_evaluation_price_rows": 10, "max_folds": 31,
    }
    protected = pd.Timestamp("2020-01-12T00:00:00Z")
    plan = make_fold_plan(daily, dates[0], protected, settings)
    assert len(plan["folds"]) == 8
    previous_end = None
    for fold in plan["folds"]:
        assert pd.Timestamp(fold["train_end_utc"]) == pd.Timestamp(
            fold["evaluation_start_utc"]
        )
        assert pd.Timestamp(fold["evaluation_end_utc"]) <= protected
        if previous_end is not None:
            assert pd.Timestamp(fold["evaluation_start_utc"]) >= previous_end
        previous_end = pd.Timestamp(fold["evaluation_end_utc"])


def test_overlapping_or_holdout_fold_rejected():
    fold = {
        "train_start_utc": "2020-01-01T00:00:00Z",
        "train_end_utc": "2020-01-03T00:00:00Z",
        "evaluation_start_utc": "2020-01-02T00:00:00Z",
        "evaluation_end_utc": "2020-01-04T00:00:00Z",
    }
    with pytest.raises(ValueError):
        validate_fold(fold, "2020-01-05T00:00:00Z")


def test_new_fold_purging_does_not_use_original_split_names():
    times = pd.date_range("2020-01-01", periods=2, freq="s", tz="UTC")
    frame = pd.DataFrame({
        "feature_timestamp_utc": times,
        "target_timestamp_utc_1s": times + pd.Timedelta(seconds=1),
        "price_valid": [True, True],
        "size_pair_valid": [True, True],
        "label_valid_1s": [True, True],
        "label_1s_pips": [0.0, 1.0],
        "mid_change_1s_pips": [0.0, 0.0],
        "bid_size": [1.0, 1.0],
        "split_1s": ["validation", "purged_train"],
    })
    mask, counts = fold_eligibility(
        frame, 1, times[0], times[-1] + pd.Timedelta(seconds=1), ["bid_size"]
    )
    assert mask.tolist() == [True, False]
    assert counts["boundary_purged_rows"] == 1


def test_cache_integrity_detects_changed_output(tmp_path):
    path = tmp_path / "generated.bin"
    path.write_bytes(b"original")
    record = _record(tmp_path, path, "test")
    _verify_files(tmp_path, [record])
    path.write_bytes(b"modified")
    with pytest.raises(RuntimeError, match="integrity"):
        _verify_files(tmp_path, [record])


def daily_losses(n=12):
    return pd.DataFrame({
        "date_utc": pd.date_range("2020-01-01", periods=n, tz="UTC"),
        "rows": np.full(n, 10),
        "candidate_abs_sum": np.full(n, 5.0),
        "reference_abs_sum": np.full(n, 10.0),
        "candidate_sq_sum": np.full(n, 5.0),
        "reference_sq_sum": np.full(n, 20.0),
    })


def test_constant_paired_differences_have_known_intervals():
    result = paired_block_interval(daily_losses(), repetitions=100)
    expected_rmse = np.sqrt(0.5) - np.sqrt(2.0)
    assert result["point_delta_mae_pips"] == pytest.approx(-0.5)
    assert result["mae_interval_low_pips"] == pytest.approx(-0.5)
    assert result["mae_interval_high_pips"] == pytest.approx(-0.5)
    assert result["point_delta_rmse_pips"] == pytest.approx(expected_rmse)


def test_bootstrap_point_estimate_uses_row_weights():
    q = daily_losses(2)
    q["rows"] = [1, 9]
    q["candidate_abs_sum"] = [0.0, 0.0]
    q["candidate_sq_sum"] = [0.0, 0.0]
    q["reference_abs_sum"] = [1.0, 27.0]
    q["reference_sq_sum"] = [1.0, 81.0]
    result = paired_block_interval(
        q, block_dates=1, repetitions=100, minimum_dates=2
    )
    assert result["point_delta_mae_pips"] == pytest.approx(-2.8)
    assert result["point_delta_rmse_pips"] == pytest.approx(-np.sqrt(8.2))


def test_resampling_is_reproducible():
    q = daily_losses()
    q["candidate_abs_sum"] += np.arange(len(q)) * 0.1
    first = paired_block_interval(q, repetitions=100, seed=42)
    second = paired_block_interval(q, repetitions=100, seed=42)
    assert first == second


def test_too_few_dates_does_not_invent_an_interval():
    result = paired_block_interval(daily_losses(3), repetitions=100)
    assert result["status"] == "insufficient_dates"
    assert result["mae_interval_low_pips"] is None
    assert result["rmse_interval_high_pips"] is None
'''.lstrip()

print("Prepared walk-forward and uncertainty tests.")

In [ ]:
for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if path.exists():
        existing = path.read_text(encoding="utf-8").replace("\r\n", "\n")
        if existing != source:
            raise RuntimeError(f"Existing source differs: {relative}. Stop.")

for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(source)
        print("CREATED", relative)
    else:
        print("REUSED", relative)

importlib.invalidate_caches()
probe = subprocess.run(
    [
        sys.executable, "-c",
        "from pathlib import Path; from eurusd_research import walk_forward; "
        "print(Path(walk_forward.__file__).resolve())",
    ],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True, text=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("Walk-forward import failed.")
if Path(probe.stdout.strip()).resolve() != (
    REPO_ROOT / "src/eurusd_research/walk_forward.py"
).resolve():
    raise RuntimeError("Imported a different project module.")

test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--color=no"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode:
    raise RuntimeError("Tests failed. Stop and return Cell 12 output.")
test_summary = test_run.stdout.strip().splitlines()[-1]

from eurusd_research.walk_forward import (
    MODEL_NAMES, _record, _verify_files,
    make_fold_plan, paired_block_interval, run_fold,
)

for relative in (
    "artifacts/manifests/walk_forward/example.json",
    "artifacts/forecasts/walk_forward/example.parquet",
    "artifacts/models/walk_forward/example.joblib",
    "reports/private/notebook06_execution.json",
    "reports/tables/walk_forward/example.csv",
    "reports/figures/walk_forward/example.png",
):
    checked = subprocess.run(
        ["git", "check-ignore", "--no-index", "-q", "--", relative],
        cwd=REPO_ROOT, capture_output=True, text=True,
    )
    if checked.returncode != 0:
        raise RuntimeError(f"Generated/private path is not ignored: {relative}")

print("PASS: imports, tests, and output ignore rules.")

In [ ]:
planning_parts = [
    part for part in feature_state["parts"]
    if utc(part["start_utc"]) < protected_start
]
planning_files = [
    str(Path(feature_state["output_dir"]) / part["path"])
    for part in planning_parts
]
if not planning_files or any(not Path(path).is_file() for path in planning_files):
    raise RuntimeError("Development feature files are missing at the current location.")

connection = feature_io._connect(
    planning_files, "features",
    REPO_ROOT / "artifacts/manifests/walk_forward/planning",
    feature_settings,
)
try:
    daily_coverage = connection.execute("""
        SELECT date_trunc('day', feature_timestamp_utc) AS date_utc,
               count(*) FILTER (WHERE price_valid) AS usable_price_rows
        FROM features
        WHERE feature_timestamp_utc < CAST(? AS TIMESTAMPTZ)
        GROUP BY 1 ORDER BY 1
    """, [protected_start.isoformat()]).fetchdf()
finally:
    connection.close()

fold_plan = make_fold_plan(
    daily_coverage, available_start, protected_start, wf_settings
)
previous_validation = baseline_run["identity"]["windows"]["validation"]

for fold in fold_plan["folds"]:
    fold["overlaps_previous_baseline_validation"] = bool(
        utc(fold["evaluation_start_utc"]) < utc(previous_validation["end_utc"])
        and utc(fold["evaluation_end_utc"]) > utc(previous_validation["start_utc"])
    )

protocol = {
    "scope": "January development walk-forward pilot",
    "feature_job_id": feature_state["job_id"],
    "label_job_id": label_state["job_id"],
    "input_manifest_sha256": input_hashes,
    "baseline_reference_run": baseline_run["run_id"],
    "regime_reference_run": regimes_run["run_id"],
    "protected_start_utc": protected_start.isoformat(),
    "horizons_seconds": HORIZONS,
    "model_settings": model_settings,
    "regime_settings": regime_settings,
    "feature_settings": feature_settings,
    "walk_forward_settings": wf_settings,
    "fold_plan": fold_plan,
    "implementation_sha256": {
        name: _sha256(REPO_ROOT / "src/eurusd_research" / f"{name}.py")
        for name in ("walk_forward", "models", "regimes", "evaluation")
    },
    "python": platform.python_version(),
    "versions": versions,
    "h001_status": "pending; not tested in this protocol",
    "full_history_final_test": "not configured; not executed",
}
run_id = hashlib.sha256(
    json.dumps(protocol, sort_keys=True).encode("utf-8")
).hexdigest()[:24]

manifest_dir = REPO_ROOT / "artifacts/manifests/walk_forward" / run_id
protocol_path = manifest_dir / "protocol.json"
run_manifest_path = manifest_dir / "run.json"

if protocol_path.exists():
    saved = json.loads(protocol_path.read_text(encoding="utf-8"))
    if saved != protocol:
        raise RuntimeError("The frozen protocol differs.")
else:
    feature_io.write_json(protocol_path, protocol)

print("FROZEN_DEVELOPMENT_PROTOCOL")
print("Run:", run_id)
print("Fold count:", len(fold_plan["folds"]))
print(pd.DataFrame(fold_plan["folds"]).to_string(index=False))
print("Skipped dates:", json.dumps(fold_plan["skipped_evaluation_dates"], indent=2))
print("Protected holdout starts:", protected_start.isoformat())
print("H001 and full-history final testing remain pending.")

In [ ]:
RUN_JANUARY_WALK_FORWARD = True

if not RUN_JANUARY_WALK_FORWARD:
    raise RuntimeError("Walk-forward execution disabled; later cells require its outputs.")
if shutil.disk_usage(REPO_ROOT).free < 6 * 1024**3:
    raise RuntimeError("Free at least 6 GiB before this job and DuckDB aggregation.")

job_started = time.perf_counter()
fold_results = []

for number, fold in enumerate(fold_plan["folds"], start=1):
    result = run_fold(
        REPO_ROOT, feature_state, label_state, fold, run_id, protocol
    )
    fold_results.append(result)
    feature_io.write_json(run_manifest_path, {
        "run_id": run_id,
        "status": "running",
        "protocol": protocol_path.relative_to(REPO_ROOT).as_posix(),
        "completed_fold_ids": [item["fold_id"] for item in fold_results],
    })
    print(
        f"Fold {number}/{len(fold_plan['folds'])}: {fold['fold_id']} "
        f"— {'verified cache' if result['cache_hit'] else 'fitted and saved'}"
    )

job_runtime = time.perf_counter() - job_started

if (
    _sha256(feature_manifest_path) != input_hashes["features"]
    or _sha256(label_manifest_path) != input_hashes["labels"]
):
    raise RuntimeError("Input manifests changed during walk-forward processing.")

print("PASS: all planned folds completed.")
print("Job runtime this execution:", round(job_runtime, 3), "seconds")

In [ ]:
forecast_records = [
    record
    for result in fold_results
    for record in result["files"]
    if record["kind"] == "forecasts"
]
_verify_files(REPO_ROOT, forecast_records)
forecast_files = [
    str(REPO_ROOT / record["path"]) for record in forecast_records
]

coverage_table = pd.DataFrame([
    record for result in fold_results for record in result["coverage"]
])
fit_table = pd.DataFrame([
    {"fold_id": result["fold_id"], **record}
    for result in fold_results for record in result["fit_records"]
])

tables = {}
daily_tables = []
bucket_tables = []
reconciliation_records = []

connection = feature_io._connect(
    forecast_files, "forecasts", manifest_dir / "aggregation", feature_settings
)
try:
    bad_prediction = " OR ".join(
        f"prediction_{name}_pips IS NULL OR NOT isfinite(prediction_{name}_pips)"
        for name in MODEL_NAMES
    )
    violations = int(connection.execute(f"""
        SELECT count(*) FROM forecasts
        WHERE feature_timestamp_utc >= CAST(? AS TIMESTAMPTZ)
           OR target_timestamp_utc >= CAST(? AS TIMESTAMPTZ)
           OR target_timestamp_utc
                <> feature_timestamp_utc + horizon_seconds * INTERVAL '1 second'
           OR source_timestamp_utc > feature_timestamp_utc
           OR snapshot_staleness_seconds < 0
           OR snapshot_staleness_seconds > ?
           OR actual_change_pips IS NULL
           OR NOT isfinite(actual_change_pips)
           OR {bad_prediction}
    """, [
        protected_start.isoformat(), protected_start.isoformat(),
        feature_settings["max_staleness_seconds"],
    ]).fetchone()[0])
    if violations:
        raise RuntimeError("Forecast timing, protection, or numeric checks failed.")

    for h in HORIZONS:
        rows, unique_rows = connection.execute("""
            SELECT count(*), count(DISTINCT feature_timestamp_utc)
            FROM forecasts WHERE horizon_seconds = ?
        """, [h]).fetchone()
        expected = int(coverage_table.loc[
            coverage_table["role"].eq("evaluation")
            & coverage_table["horizon_seconds"].eq(h),
            "matched_rows",
        ].sum())
        if rows != unique_rows or rows != expected:
            raise RuntimeError("Forecast uniqueness or coverage reconciliation failed.")
        reconciliation_records.append({
            "horizon_seconds": h,
            "forecast_rows": int(rows),
            "unique_feature_timestamps": int(unique_rows),
            "expected_matched_rows": expected,
        })

    groupings = {
        "overall": ["horizon_seconds"],
        "by_fold": ["horizon_seconds", "fold_id"],
        "by_year": ["horizon_seconds", "year_utc"],
        "by_session": ["horizon_seconds", "session_label"],
        "by_regime": ["horizon_seconds", "liquidity_regime"],
        "by_snapshot_age": ["horizon_seconds", "snapshot_age_band"],
        "by_endpoint": ["horizon_seconds", "same_source_record"],
    }

    for table_name, groups in groupings.items():
        pieces = []
        group_sql = ", ".join(groups)
        for model in MODEL_NAMES:
            prediction = f"prediction_{model}_pips"
            error = f"({prediction} - actual_change_pips)"
            piece = connection.execute(f"""
                SELECT {group_sql},
                       count(*) AS rows,
                       avg(abs({error})) AS mae_pips,
                       sqrt(avg({error} * {error})) AS rmse_pips,
                       avg(actual_change_pips) AS mean_actual_pips,
                       avg({prediction}) AS mean_forecast_pips,
                       avg({error}) AS forecast_bias_pips,
                       median(spread_pips) AS median_spread_pips,
                       median(abs({prediction})) AS median_abs_forecast_pips,
                       median(CASE WHEN spread_pips > 0
                           THEN abs({prediction}) / spread_pips END)
                           AS median_abs_forecast_over_spread,
                       CASE WHEN max({prediction}) - min({prediction}) > 1e-12
                                 AND var_pop({prediction}) > 0
                           THEN covar_pop({prediction}, actual_change_pips)
                                / var_pop({prediction})
                       END AS calibration_slope
                FROM forecasts
                GROUP BY {group_sql}
                ORDER BY {group_sql}
            """).fetchdf()
            piece["calibration_intercept_pips"] = (
                piece["mean_actual_pips"]
                - piece["calibration_slope"] * piece["mean_forecast_pips"]
            )
            piece.insert(0, "model", model)
            pieces.append(piece)
        tables[table_name] = pd.concat(pieces, ignore_index=True)

    for model in MODEL_NAMES:
        prediction = f"prediction_{model}_pips"
        error = f"({prediction} - actual_change_pips)"
        daily = connection.execute(f"""
            SELECT horizon_seconds,
                   date_trunc('day', feature_timestamp_utc) AS date_utc,
                   count(*) AS rows,
                   sum(abs({error})) AS abs_sum,
                   sum({error} * {error}) AS sq_sum
            FROM forecasts
            GROUP BY 1, 2 ORDER BY 1, 2
        """).fetchdf()
        daily.insert(0, "model", model)
        daily_tables.append(daily)

        buckets = connection.execute(f"""
            SELECT horizon_seconds, bucket_{model} AS bucket,
                   count(*) AS rows,
                   avg({prediction}) AS mean_forecast_pips,
                   avg(actual_change_pips) AS mean_actual_pips
            FROM forecasts
            GROUP BY 1, 2 ORDER BY 1, 2
        """).fetchdf()
        buckets.insert(0, "model", model)
        bucket_tables.append(buckets)
finally:
    connection.close()

daily_loss_table = pd.concat(daily_tables, ignore_index=True)
bucket_table = pd.concat(bucket_tables, ignore_index=True)
reconciliation_table = pd.DataFrame(reconciliation_records)
overall_table = tables["overall"]

for h in HORIZONS:
    selected = overall_table["horizon_seconds"].eq(h)
    group = overall_table.loc[selected]
    if group["rows"].nunique() != 1:
        raise RuntimeError("Model evaluation populations differ.")
    zero = group.loc[group["model"].eq("zero")].iloc[0]
    overall_table.loc[selected, "mae_gain_vs_zero_pips"] = (
        zero["mae_pips"] - overall_table.loc[selected, "mae_pips"]
    )
    overall_table.loc[selected, "rmse_gain_vs_zero_pips"] = (
        zero["rmse_pips"] - overall_table.loc[selected, "rmse_pips"]
    )

print("PASS: unique forecasts, protected boundaries, and coverage reconciliation.")
print("WALK_FORWARD_METRICS")
print(overall_table[[
    "horizon_seconds", "model", "rows", "mae_pips", "rmse_pips",
    "mae_gain_vs_zero_pips", "rmse_gain_vs_zero_pips",
]].to_string(index=False))

In [ ]:
uncertainty_started = time.perf_counter()
interval_records = []

comparisons = [
    ("ridge_price", "zero"),
    ("ridge_price_size", "zero"),
    ("ridge_price_size", "ridge_price"),
    ("trailing", "zero"),
]

for h in HORIZONS:
    for candidate, reference in comparisons:
        left = daily_loss_table.loc[
            daily_loss_table["horizon_seconds"].eq(h)
            & daily_loss_table["model"].eq(candidate),
            ["date_utc", "rows", "abs_sum", "sq_sum"],
        ]
        right = daily_loss_table.loc[
            daily_loss_table["horizon_seconds"].eq(h)
            & daily_loss_table["model"].eq(reference),
            ["date_utc", "rows", "abs_sum", "sq_sum"],
        ]
        paired = left.merge(
            right, on="date_utc", how="outer",
            suffixes=("_candidate", "_reference"),
            validate="one_to_one",
        ).sort_values("date_utc")
        if (
            paired.isna().any().any()
            or not paired["rows_candidate"].eq(paired["rows_reference"]).all()
        ):
            raise RuntimeError("Paired daily comparison populations differ.")

        losses = pd.DataFrame({
            "date_utc": paired["date_utc"],
            "rows": paired["rows_candidate"],
            "candidate_abs_sum": paired["abs_sum_candidate"],
            "reference_abs_sum": paired["abs_sum_reference"],
            "candidate_sq_sum": paired["sq_sum_candidate"],
            "reference_sq_sum": paired["sq_sum_reference"],
        })

        for block_dates in wf_settings["bootstrap_block_dates"]:
            result = paired_block_interval(
                losses,
                block_dates=block_dates,
                repetitions=wf_settings["bootstrap_repetitions"],
                confidence=wf_settings["confidence"],
                seed=wf_settings["random_seed"],
                minimum_dates=wf_settings["minimum_bootstrap_dates"],
            )
            interval_records.append({
                "horizon_seconds": h,
                "candidate": candidate,
                "reference": reference,
                **result,
            })

interval_table = pd.DataFrame(interval_records)
size_interval_table = interval_table.loc[
    interval_table["candidate"].eq("ridge_price_size")
    & interval_table["reference"].eq("ridge_price")
].copy()

uncertainty_runtime = time.perf_counter() - uncertainty_started

print("SIZE_MINUS_PRICE_BLOCK_INTERVALS")
print(size_interval_table.to_string(index=False))
print("Negative differences favor added size.")
print("Intervals are exploratory; subgroup uncertainty is not estimated.")

In [ ]:
import matplotlib.pyplot as plt

fold_dates = pd.DataFrame([
    {
        "fold_id": fold["fold_id"],
        "evaluation_date": pd.Timestamp(fold["evaluation_start_utc"]),
    }
    for fold in fold_plan["folds"]
])
by_fold = tables["by_fold"].merge(fold_dates, on="fold_id", validate="many_to_one")

figure, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for axis, h in zip(axes.flat, HORIZONS):
    subset = by_fold.loc[by_fold["horizon_seconds"].eq(h)]
    pivot = subset.pivot(
        index="evaluation_date", columns="model", values="rmse_pips"
    ).sort_index()
    for model, label in (
        ("ridge_price", "Price Ridge"),
        ("ridge_price_size", "Price + size Ridge"),
    ):
        axis.plot(
            pivot.index, pivot["zero"] - pivot[model],
            marker="o", markersize=3, label=label,
        )
    axis.axhline(0, color="grey", linewidth=1)
    axis.set_title(f"{h}-second horizon")
    axis.set_ylabel("RMSE gain versus zero (pips)")
    axis.tick_params(axis="x", rotation=35)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)

figure.suptitle("January development walk-forward stability")
figure_path = (
    REPO_ROOT / "reports/figures" / f"walk-forward-stability-{run_id}.png"
)
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=160)
plt.show()
plt.close(figure)

print("Saved:", figure_path.relative_to(REPO_ROOT).as_posix())

In [ ]:
def relative_path(path):
    return Path(path).resolve().relative_to(REPO_ROOT).as_posix()


def update_section(path, name, content):
    begin = f"<!-- {name}:BEGIN -->"
    finish = f"<!-- {name}:END -->"
    existing = path.read_text(encoding="utf-8") if path.exists() else ""
    if existing.count(begin) != existing.count(finish) or existing.count(begin) > 1:
        raise RuntimeError(f"Malformed generated section: {path.name}")
    block = f"{begin}\n{content.strip()}\n{finish}"
    if begin in existing:
        left, remainder = existing.split(begin, 1)
        _, right = remainder.split(finish, 1)
        text = left + block + right
    else:
        text = existing.rstrip() + "\n\n" + block + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(text)


if (
    _sha256(feature_manifest_path) != input_hashes["features"]
    or _sha256(label_manifest_path) != input_hashes["labels"]
):
    raise RuntimeError("Input manifests changed.")

table_dir = REPO_ROOT / "reports/tables" / run_id
table_dir.mkdir(parents=True, exist_ok=True)

tables.update({
    "coverage": coverage_table,
    "fit_boundaries": fit_table,
    "daily_loss_totals": daily_loss_table,
    "forecast_buckets": bucket_table,
    "paired_block_intervals": interval_table,
    "forecast_reconciliation": reconciliation_table,
    "fold_plan": pd.DataFrame(fold_plan["folds"]),
})

table_paths = {}
report_records = []
for name, table in tables.items():
    path = table_dir / f"{name}.csv"
    table.to_csv(path, index=False)
    table_paths[name] = relative_path(path)
    report_records.append(_record(REPO_ROOT, path, "table"))

example_path = REPO_ROOT / "configs/research_config.example.json"
local_path = REPO_ROOT / "configs/research_config.local.json"
example = json.loads(example_path.read_text(encoding="utf-8-sig"))
local = json.loads(local_path.read_text(encoding="utf-8-sig"))

if "walk_forward" in local and local["walk_forward"] != wf_settings:
    raise RuntimeError("A conflicting local walk-forward specification needs review.")
example["walk_forward"] = wf_settings
local.pop("walk_forward", None)
local["active_walk_forward_manifest"] = relative_path(run_manifest_path)

feature_io.write_json(example_path, example)
feature_io.write_json(local_path, local)

environment_path = REPO_ROOT / "reports/private/validated_walk_forward_environment.json"
feature_io.write_json(environment_path, {
    "python": platform.python_version(),
    "packages": versions,
    "validation": test_summary,
})

note = f"""
### Notebook 06 — January development walk-forward pilot

Implemented rolling chronological folds using the preceding
{wf_settings["training_calendar_days"]} calendar days for training and
one following UTC day for evaluation.

Each fold independently fits preprocessing, Ridge models, regime thresholds,
and training-prediction bucket boundaries. Label intervals reaching fold
boundaries are purged.

Model feature families use identical training and evaluation observations.
Completed folds are resumable with artifact integrity checks.

Reports include aggregate and date/session/regime/freshness diagnostics,
calibration, forecast buckets, spread-scale comparisons, and paired
moving-block intervals using ordered evaluation-date loss totals.

Intervals are exploratory given the short development sample and assumptions
about time dependence. Subgroup intervals are not estimated.

Executed correctness checks: {test_summary}.
Forecast uniqueness, coverage, chronology, and protected-boundary checks passed.
Notebook 06 owner review remains pending.

The reserved pilot holdout was not evaluated.
This is not full-history final-test validation.
Broader ingestion, multi-year stability, final-test dates, H001 testing,
and sustained-wide-spread episode analysis remain pending.
"""

for relative in ("README.md", "reports/research_results.md"):
    update_section(REPO_ROOT / relative, "NOTEBOOK_06", note)

update_section(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK_06",
    f"""
### Notebook 06

- Raymond Fendi requested the walk-forward evaluation notebook.
- OpenAI Codex supplied fold orchestration, paired uncertainty calculations,
  tests, and documentation.
- Executed correctness checks: {test_summary}.
- Per-fold models and regime thresholds were fitted using training data only.
- Notebook 06 owner review and understanding checkpoint: pending.
- H001 remains pending; no dedicated H001 experiment was executed.
- Full-history final testing remains pending.
""",
)

private_report_path = REPO_ROOT / "reports/private" / f"walk-forward-{run_id}.md"
private_report_path.parent.mkdir(parents=True, exist_ok=True)
private_report_path.write_text(
    "# January development walk-forward evaluation\n\n"
    "Reserved holdout not evaluated. H001 not tested.\n\n"
    "## Aggregate metrics\n\n```\n"
    + overall_table.to_string(index=False)
    + "\n```\n\n## Size-minus-price paired intervals\n\n```\n"
    + size_interval_table.to_string(index=False)
    + "\n```\n\n"
    "Intervals are exploratory and conditional on the saved development forecasts.\n",
    encoding="utf-8",
)

run_state = {
    "run_id": run_id,
    "status": "complete",
    "protocol": relative_path(protocol_path),
    "fold_manifests": [result["manifest_path"] for result in fold_results],
    "completed_fold_ids": [result["fold_id"] for result in fold_results],
    "reports": report_records,
    "validation": {
        "pytest": test_summary,
        "forecast_reconciliation": "passed",
        "protected_boundary_checks": "passed",
        "input_manifests_unchanged": "passed",
        "pilot_holdout_evaluated": False,
        "owner_review": "pending",
    },
}
feature_io.write_json(run_manifest_path, run_state)

print("Saved walk-forward reports and active run selection.")
print("Reserved holdout and H001 remain untouched.")

In [ ]:
def json_records(frame):
    return json.loads(frame.to_json(orient="records", date_format="iso"))


all_fold_files = [
    record for result in fold_results for record in result["files"]
]
evaluation_dates = sorted({
    fold["evaluation_start_utc"] for fold in fold_plan["folds"]
})
evaluation_clock_rows = sum(
    int((
        utc(fold["evaluation_end_utc"])
        - utc(fold["evaluation_start_utc"])
    ).total_seconds())
    for fold in fold_plan["folds"]
)

summary = {
    "notebook": NOTEBOOK,
    "run_id": run_id,
    "scope": "January development walk-forward pilot",
    "python_version": platform.python_version(),
    "package_versions": versions,
    "configuration": {
        "feature_job_id": feature_state["job_id"],
        "label_job_id": label_state["job_id"],
        "walk_forward_settings": wf_settings,
        "horizons_seconds": HORIZONS,
        "ridge_alpha": model_settings["ridge_alpha"],
        "price_predictor_count": len(model_settings["price_features"]),
        "added_size_predictor_count": len(model_settings["size_features"]),
        "maximum_snapshot_age_seconds": feature_settings["max_staleness_seconds"],
        "volatility_return_interval_seconds": 1,
        "regime_thresholds": "refitted within each training fold",
        "parameter_search": "none",
    },
    "folds": {
        "count": len(fold_results),
        "first_evaluation_start_utc": evaluation_dates[0],
        "last_evaluation_end_utc": fold_plan["folds"][-1]["evaluation_end_utc"],
        "evaluation_dates": len(evaluation_dates),
        "evaluation_clock_rows": evaluation_clock_rows,
        "skipped_dates": fold_plan["skipped_evaluation_dates"],
        "overlap_previous_baseline_validation_folds": sum(
            fold["overlaps_previous_baseline_validation"]
            for fold in fold_plan["folds"]
        ),
        "folds_fitted_this_execution": sum(
            not result["cache_hit"] for result in fold_results
        ),
        "folds_reused_after_verification": sum(
            result["cache_hit"] for result in fold_results
        ),
        "protected_holdout_start_utc": protected_start.isoformat(),
    },
    "forecast_reconciliation": json_records(reconciliation_table),
    "overall_metrics": json_records(overall_table),
    "size_minus_price_intervals": json_records(size_interval_table),
    "validation": {
        "imports_from_notebooks_directory": "passed",
        "pytest": test_summary,
        "fold_specific_target_purging": "implemented and checked",
        "training_only_models_and_regimes": "implemented with saved fit boundaries",
        "forecast_uniqueness_and_coverage": "passed",
        "protected_boundary_checks": "passed",
        "input_manifests_unchanged": "passed",
        "pilot_holdout_evaluated": False,
        "full_history_final_test_evaluated": False,
        "owner_review": "pending",
    },
    "measurements": {
        "job_seconds_this_execution": round(job_runtime, 3),
        "uncertainty_seconds": round(uncertainty_runtime, 3),
        "saved_model_count": sum(
            item["kind"] == "model" for item in all_fold_files
        ),
        "forecast_parquet_bytes": sum(
            item["bytes"] for item in all_fold_files
            if item["kind"] == "forecasts"
        ),
        "maximum_observed_training_frame_bytes": max(
            result["observed_input_frame_bytes"]["train"]
            for result in fold_results
        ),
        "maximum_observed_evaluation_frame_bytes": max(
            result["observed_input_frame_bytes"]["evaluation"]
            for result in fold_results
        ),
        "notebook_elapsed_seconds": round(time.perf_counter() - notebook_started, 3),
    },
    "warnings": [
        "Only the processed January development sample is evaluated.",
        "Some dates overlap previously inspected baseline validation data.",
        "Overlapping training windows reuse observations; training counts are not unique samples.",
        "Evaluation coverage excludes explicitly skipped dates.",
        "Paired intervals are exploratory and conditional on saved forecasts.",
        "Two- and three-date blocks may not capture all dependence or nonstationarity.",
        "Multiple comparisons are reported without confirmatory significance claims.",
        "Year reporting currently contains only the available processed year.",
        "Subgroup uncertainty intervals are not estimated.",
        "Displayed-size units and true side-specific quote ages remain unresolved.",
        "Dataframe measurements are not total process peak memory.",
        "H001 and sustained-wide-spread episode testing remain pending.",
        "Full-history scaling, final-test dates, and final testing remain pending.",
        "No achievable-profit conclusion has been established.",
    ],
    "saved_outputs": {
        "active_walk_forward_manifest": relative_path(run_manifest_path),
        "frozen_development_protocol": relative_path(protocol_path),
        "tables": table_paths,
        "stability_figure": relative_path(figure_path),
        "private_report": relative_path(private_report_path),
        "validated_environment": relative_path(environment_path),
        "execution_summary": "reports/private/notebook06_execution.json",
        "hypothesis_record": "reports/research_hypotheses.md",
    },
}

feature_io.write_json(
    REPO_ROOT / "reports/private/notebook06_execution.json", summary
)
print("NOTEBOOK_06_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))